# CADP compression of the clean AIS

Trajectory compression keeps the points of a track that matter and drops the rest. This notebook
compresses **all** the clean AIS of the chosen dataset with **CADP**, the Context-Aware
Douglas-Peucker method of

> Mehri, S., Hooshangi, N., Mahdizadeh Gharakhanlou, N. (2025). *A Novel Context-Aware Douglas-Peucker (CADP) Trajectory Compression Method.* ISPRS Int. J. Geo-Inf. 14(2), 58. [doi:10.3390/ijgi14020058](https://doi.org/10.3390/ijgi14020058)

and saves the result in the same format as notebook 03 (2stage-CPLS), so the two can be compared.

The figures are interactive (hvplot / bokeh): they work in Jupyter and on [nbviewer](https://nbviewer.org/github/smehri/CATraj/blob/main/notebooks/04_cadp_compression.ipynb), but GitHub's notebook viewer does not run JavaScript, so they do not show on github.com.

1. **The method**: Douglas-Peucker, the context it is extended with, the parameters
2. **Load** the clean AIS split into trips, with the same settings as notebook 03
3. **Context**: the sea state (bottom depth, wave height, wave direction) annotated on every position
4. **Compress**: `compress` over every trip: plain Douglas-Peucker, the paper's three single-context CADP variants (depth, wave height, wave direction), and our combined run
5. **Save**: one `compressed_ais_<variant>.parquet` per variant and a summary table
6. **Results**: compression ratio overall and by vessel type; against Douglas-Peucker and 2stage-CPLS
7. **Vessels before and after**, and **how faithfully the context survived**

**Inputs**: `AIS/ais.parquet` and `Ocean/ocean_<month>.parquet` of the preprocessed folder (written
by `01_preprocessing.ipynb`). **Outputs**:
`results/04_cadp_compression/<DATA>/` (the trips, `trips_ais.parquet`, and the compressed files).
The package is built on [MovingPandas](https://movingpandas.org):

> Graser, A. (2019). *MovingPandas: Efficient Structures for Movement Data in Python.* GI_Forum 2019, 7(1), 54-68. [doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54)

## Data

The notebook compresses the preprocessed AIS (`ais.parquet`: cleaned, median-filtered positions of every
vessel, with speed, course, status and vessel type) and annotates it with the preprocessed **sea state**
(`ocean_<month>.parquet`: a 3-hourly WAVEWATCH III grid of bottom depth, wave height and wave direction).

By default it runs on the **committed, preprocessed two-day sample** (2015-11-09 to 2015-11-11 UTC, Brest
area, 565 vessels). To process the **full archive**, run `notebooks/01_preprocessing.ipynb` first (its
download and preprocessing sections), then set `DATA = "full"` below, or run the notebook with the
environment variable `CATRAJ_DATA=full`. The full archive covers October to December 2015 and February to
March 2016. **January 2016 is left out of every dataset**: the Zenodo archive has no sea state for it (its
`oc_january.csv` is a byte copy of `oc_november.csv`), so those AIS positions could not be annotated with
the context CADP needs and are not used anywhere.

Data: Ray, C. et al. (2019), *Heterogeneous integrated dataset for Maritime Intelligence, Surveillance, and
Reconnaissance*, Data in Brief 25, 104141, [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141);
Zenodo [doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595),
record <https://zenodo.org/records/1167595>. AIS from NARI, Brest (CC BY-NC-SA 4.0: anything derived from it,
including this notebook's outputs, is shared under the same licence). Sea state: SHOM-IFREMER WAVEWATCH III
hindcast, citation mandatory: Boudiere, E. et al. (2013), *A suitable metocean hindcast database for the design
of Marine energy converters*, Int. J. Marine Energy 3-4, e40-e52,
[doi:10.1016/j.ijome.2013.11.010](https://doi.org/10.1016/j.ijome.2013.11.010).

In [1]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")   # see catraj/__init__.py

DATA = os.environ.get("CATRAJ_DATA", "sample")      # "sample" or "full"
N_JOBS = int(os.environ.get("CATRAJ_N_JOBS", "-1"))  # -1: every core; 1: serial

import time
from pathlib import Path

import holoviews as hv
import hvplot.pandas  # noqa: F401  -- registers .hvplot
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
from pyproj import Transformer

hv.extension("bokeh")                  # interactive figures: zoom, pan, hover, click a legend entry

from catraj import parallel_map
from catraj.annotation import DLT, IDW, annotate_points
from catraj.compression import compress, compression_ratio, context_errors
from catraj.datasets import load_coastline, repo_root, sample_root
from catraj.preprocessing import split_trips

if DATA not in ("sample", "full"):
    raise ValueError(f"DATA must be 'sample' or 'full', not {DATA!r}")
ROOT = repo_root()
PRE = sample_root() if DATA == "sample" else ROOT / "data" / "ZenodoData" / "preprocessed"
OUT = ROOT / "results" / "04_cadp_compression" / DATA
OUT.mkdir(parents=True, exist_ok=True)
NB03 = ROOT / "results" / "03_two_stage_cpls_compression" / DATA     # notebook 03's outputs, if run (only used for the 2stage-CPLS comparison, and only when it ran on the same positions)

# ---- the parameters of CADP (the paper's experiment, Mehri et al. 2025) ---------
EPSILON = 1_000.0                    # Douglas-Peucker threshold, metres
THRESHOLDS = {                       # one per context variable, in its own units
    "bottom_depth": 80.0,            # m
    "wave_height": 0.15,             # m
    "wave_direction": 1.5,           # degrees, on the circle
}
CIRCULAR = ["wave_direction"]        # bearings: differenced on the circle
MODE = "post"                        # the method as published (see section 1)
# The paper's variants: Douglas-Peucker plus ONE context at a time. COMBINED (all three at once) is our addition.
VARIANTS = {"depth": "bottom_depth", "wave_height": "wave_height", "wave_direction": "wave_direction"}
# ---- the trips (the same settings as notebook 03) -----------------------------------------
MAX_GAP = 1_800.0        # a gap of over 30 minutes between two reports starts a new trip
MIN_TRIP_POINTS = 10     # trips with fewer positions are dropped
# ---- the iterative mode of the combined run, for comparison: cheap on the sample, optional on the full archive
RUN_ITERATIVE = os.environ.get("CATRAJ_RUN_ITERATIVE", "1" if DATA == "sample" else "0") == "1"

print(f"DATA = {DATA!r}, N_JOBS = {N_JOBS}\ninput:  {PRE}\noutput: {OUT}")

DATA = 'sample', N_JOBS = -1
input:  C:\dev\CATraj\data\ZenodoData\sample\preprocessed
output: C:\dev\CATraj\results\04_cadp_compression\sample


## Parallel processing

Three steps of this notebook run in parallel, all through the package's `n_jobs`
(`catraj.parallel_map` / `map_groups`, `src/catraj/_parallel.py`):

* **`annotate_points(..., n_jobs=N_JOBS)`** (section 3): the positions are grouped into *time blocks* of a few
  sea-state epochs, each with a small margin of epochs either side; the blocks are independent and are
  shared out over the processes.
* **`compress(..., by="trip_id", n_jobs=N_JOBS)`** (section 4, for CADP and for plain Douglas-Peucker):
  every trip is compressed independently of the others (Douglas-Peucker, the context errors and the union
  only ever look at one trip), so the trips are shared out. They are grouped into chunks of about equal
  number of positions, because the cost depends on a trip's length.
* **`split_trips(..., n_jobs=N_JOBS)`**, when the trips have to be recomputed (section 2), and the
  **context-fidelity check** (section 7, `catraj.parallel_map` over the trips): one task per trip.

`N_JOBS` follows the scikit-learn convention: `1` runs everything serially in this process, `-1` uses every
core, `k` uses `k` processes. The results come back in task order and every trip is computed by the same
code either way, so **parallel and serial runs give identical results**; only the wall time changes. Each
worker receives its share of the trips (pickled), so on the full archive the memory in use is roughly the
data frame in this process plus the chunks in flight in the workers; the annotation works one calendar
month at a time so the sea-state grid of the whole archive is never in memory. To run serially, set
`N_JOBS = 1` or start Jupyter with `CATRAJ_N_JOBS=1`.

## 1 · The method

**Douglas-Peucker** (DP) keeps the points that carry *spatial* information: a point is dropped when it lies
within `EPSILON` of the line between its neighbours. A point on a straight leg is spatially redundant, and yet
it may be the moment the waves rose or the water shoaled; once it is dropped, that information is gone from
every analysis downstream. **CADP** (Algorithm 2 of the paper) therefore runs in three steps:

1. **Spatial.** Simplify the trajectory with plain Douglas-Peucker at `EPSILON`.
2. **Context.** For every dropped point *i* between two consecutive retained points *a* and *b*, take the
   context the simplified trajectory implies at its time, the linear interpolation in *time*

   `c_hat_i = c_a + (c_b - c_a) * (t_i - t_a) / (t_b - t_a)`,   `e_i = | c_i - c_hat_i |`,

   and compare it with the context actually observed. If `e_i` exceeds that variable's threshold, point *i*
   is added back. This is the spatial error of Eq. 1 carried over to context: the projection onto the
   simplifying segment becomes the interpolation of the context along it, in the variable's own units.
3. **Union.** The result is the DP points plus every point added in step 2.

**Which context.** The paper's experiments run CADP with **one context variable at a time** (its Table 2
lists "Spatial + water depth", "Spatial + wave height" and "Spatial + water direction" as separate compressed
datasets, and its Table 1 varies only the wave-direction threshold). The context is the **sea state** met along
the way, three **environmental** variables annotated on every position from the WAVEWATCH III grid (section 3),
each with the paper's threshold:

| variant (file suffix) | variable | meaning | threshold | note |
|---|---|---|---|---|
| `depth` | `bottom_depth` | bottom depth | 80 m | static in time; changes only as the vessel moves |
| `wave_height` | `wave_height` | significant wave height | 0.15 m | |
| `wave_direction` | `wave_direction` | mean wave direction, a **bearing** | 1.5 degrees | differenced and interpolated on the circle: 350 and 10 degrees are 20 degrees apart |

with `EPSILON` = 1,000 m. These three runs are the **main results** of this notebook, each saved to its own
file. As **our own addition**, clearly not part of the paper's experiments, a fourth run applies all three
variables together (`combined`): the package tests each variable on its own and adds a point if **any** is over its
threshold, so it keeps at least the union of the three single-context sets and typically many more. The package
handles the other two domains of the paper the same way (`thresholds` also takes *internal* variables such as
`speed_over_ground`, and `nominal` takes *semantic* ones such as the navigational status); this notebook keeps to
the sea state.

**Reference points from the paper** (different data: its own AIS and a different period; so these are
orientation, not a reproduction). Table 1, CADP with wave direction: R = 70.47 % at 1 degree, 72.15 % at 1.5
degrees, 72.84 % at 2 degrees, 73.25 % at 2.5 degrees. In the text, on its data, Douglas-Peucker at 1 km gives
R = 76.78 % and 2stage-CPLS R = 72.08 %. The paper's Table 2 compares the same datasets by the validation RMSE of an
LSTM, not by R.

**Two modes.** `mode="post"` is the method as published: every error is measured once, against the DP
trajectory, and every point over its threshold is added at once. That is the mode this notebook uses and saves.
`mode="iterative"` re-measures as the trajectory grows (only each segment's worst point is added per round), so
every dropped point ends up within its thresholds of its final neighbours, usually with far fewer points; the
notebook compares the two on the combined run on the sample. `help(catraj.compression.cadp)` lists every decision.

**Filtering.** `compress` median-filters the positions and the threshold columns over 30 s by default. The
preprocessed AIS **already is** filtered (notebook 01 applies the same 30 s median) and the sea state is a
smooth 3-hourly field interpolated to the positions, so filtering again would only smooth smooth data, change the
positions later steps read and cost time. We pass `filter_window=None`: the compressed rows are then exactly
rows of the clean data, with their original values.

## 2 · Load the clean AIS as trips

The AIS is split into **trips** with the same settings as notebook 03 (`catraj.preprocessing.split_trips`): each
vessel's track is cut wherever two consecutive reports are **more than `MAX_GAP` = 30 minutes** apart (the
vessel was out of range, so a chord across the gap would invent a path), with no stay rule, and trips of fewer
than `MIN_TRIP_POINTS` = 10 positions are dropped. Every trip has a `trip_id`, unique over the dataset.

The trips are always computed here, from this dataset's preprocessed AIS, with the same code and the same two
numbers, and saved to `trips_ais.parquet` in this notebook's results folder.

In [2]:
COLUMNS = ["mmsi", "trip_id", "unix_time", "x", "y", "lon", "lat", "speed_over_ground",
           "course_over_ground", "vessel_type", "navigational_status", "true_heading",
           "rate_of_turn", "ship_type_code"]          # the columns of notebook 03's outputs

t0 = time.perf_counter()
N_ALL = N_TRIPS_ALL = N_TINY = N_TINY_POINTS = None
from catraj.datasets import load_ais
ais = load_ais(PRE)
N_ALL = len(ais)
ais = split_trips(ais, by="mmsi", time_column="unix_time", max_gap=MAX_GAP,
                  stay_radius=None, stay_duration=None, stay_column=None,
                  trip_column="trip", n_jobs=N_JOBS)
sizes = ais.groupby(["mmsi", "trip"], sort=False).size()
small = sizes < MIN_TRIP_POINTS
N_TRIPS_ALL, N_TINY, N_TINY_POINTS = len(sizes), int(small.sum()), int(sizes[small].sum())
big = sizes[~small].index.to_frame(index=False)
trips = ais.merge(big, on=["mmsi", "trip"])
trips = trips.sort_values(["mmsi", "unix_time"], kind="stable").reset_index(drop=True)
trips["trip_id"] = trips.groupby(["mmsi", "trip"], sort=False).ngroup().astype("int64")
del ais
SPLIT_S = time.perf_counter() - t0
SOURCE = "computed here with split_trips"
trips = pd.DataFrame(trips[COLUMNS]).sort_values(["mmsi", "unix_time"], kind="stable").reset_index(drop=True)
N = len(trips)
N_TRIPS = trips.trip_id.nunique()
trips.to_parquet(OUT / "trips_ais.parquet", index=False)
print(f"trips {SOURCE}  ({time.perf_counter() - t0:.1f} s)")
if N_ALL is not None:
    print(f"split into {N_TRIPS_ALL:,} trips; dropped {N_TINY:,} tiny trips of fewer than {MIN_TRIP_POINTS} "
          f"positions ({N_TINY_POINTS:,} positions, {N_TINY_POINTS / N_ALL:.2%} of the data)")
print(f"{N_TRIPS:,} trips, {N:,} positions, {trips.mmsi.nunique():,} vessels, "
      f"{pd.to_datetime(trips.unix_time.min(), unit='s'):%Y-%m-%d} to "
      f"{pd.to_datetime(trips.unix_time.max(), unit='s'):%Y-%m-%d} UTC")
print("positions per trip:", trips.groupby("trip_id").size().describe().round(0).astype(int).to_dict())

trips computed here with split_trips  (5.2 s)
split into 2,213 trips; dropped 1,650 tiny trips of fewer than 10 positions (3,979 positions, 1.68% of the data)
563 trips, 233,339 positions, 253 vessels, 2015-11-09 to 2015-11-10 UTC
positions per trip: {'count': 563, 'mean': 414, 'std': 1671, 'min': 10, '25%': 17, '50%': 36, '75%': 143, 'max': 17461}


## 3 · Context: the sea state at every position

CADP reads its context from ordinary columns, so the sea state is first **annotated** on the positions
(`catraj.annotation.annotate_points`, covered in `02_trajectory_annotation.ipynb`). The ocean grid is
3-hourly, in longitude and latitude (here projected to UTM 30N so that distances are in metres):

* **Spatially**, each variable is interpolated from the grid to the position by inverse-distance weighting
  over the 4 nearest cells (`IDW(k=4)`); for a bearing on the unit circle (`circular=True`), so that 350 and 10
  degrees average to 0, not 180.
* **Through time**, between the two 3-hourly epochs that bracket the position's time, linearly (`DLT`).
* **Bottom depth** does not change in time, so one epoch's grid is enough and only the spatial stage is applied.

The grid is stored one file per month, and millions of positions times many epochs would not fit in
memory at once, so the annotation works **one calendar month at a time**: the month's positions, and the grid
rows of the month plus a margin of two epochs, are read, annotated (with `n_jobs`) and released. The context
is already filtered by preprocessing (`filter_window=None`).

In [3]:
OCEAN_COLUMNS = ["unix_time", "lon", "lat", "bottom_depth", "wave_height", "wave_direction"]
MARGIN = 2 * 3 * 3600                      # grid rows read this far either side of a month
to_utm = Transformer.from_crs(4326, 32630, always_xy=True)

def read_ocean(lo, hi):
    # Grid rows with lo <= unix_time <= hi from every monthly file, cells projected to UTM 30N.
    files = sorted((PRE / "Ocean").glob("ocean_*.parquet"))
    parts = [pd.read_parquet(f, columns=OCEAN_COLUMNS,
                             filters=[("unix_time", ">=", lo), ("unix_time", "<=", hi)]) for f in files]
    oc = pd.concat([p for p in parts if len(p)], ignore_index=True)
    oc = oc.drop_duplicates(["unix_time", "lon", "lat"], ignore_index=True)   # months overlap by an epoch
    oc["x"], oc["y"] = to_utm.transform(oc.lon.to_numpy(), oc.lat.to_numpy())
    return oc

def stations(oc, column):
    # The table annotate_points takes: x, y, t, value.
    ok = oc[column].notna()
    return pd.DataFrame({"x": oc.x[ok].to_numpy(), "y": oc.y[ok].to_numpy(),
                         "t": oc.unix_time[ok].to_numpy(float), "value": oc[column][ok].to_numpy(float)})

month = pd.to_datetime(trips.unix_time, unit="s").dt.strftime("%Y-%m")
for name in THRESHOLDS:
    trips[name] = np.nan
rows = []
T0 = time.perf_counter()
for label in sorted(month.unique()):
    t0 = time.perf_counter()
    sel = np.flatnonzero((month == label).to_numpy())
    pts = trips.iloc[sel]
    x, y, t = pts.x.to_numpy(), pts.y.to_numpy(), pts.unix_time.to_numpy(float)
    oc = read_ocean(int(t.min()) - MARGIN, int(t.max()) + MARGIN)
    grid = oc[oc.unix_time == oc.unix_time.min()]            # depth: one epoch, spatial stage only
    grid = grid[grid.bottom_depth.notna()]
    values = {"bottom_depth": IDW(k=4).fit(grid.x, grid.y, grid.bottom_depth)(x, y)}
    for var in ("wave_height", "wave_direction"):
        circ = var in CIRCULAR
        values[var] = annotate_points(stations(oc, var), x, y, t,
                                      spatial_method=IDW(k=4, circular=circ),
                                      temporal_method=DLT(circular=circ), circular=circ,
                                      n_jobs=N_JOBS, filter_window=None)
    for var, v in values.items():
        trips.loc[trips.index[sel], var] = v
    rows.append({"month": label, "positions": len(sel), "epochs": oc.unix_time.nunique(),
                 **{f"{v}_annotated": float(np.isfinite(values[v]).mean()) for v in THRESHOLDS},
                 "seconds": time.perf_counter() - t0})
    del oc, grid
ANNOTATE_S = time.perf_counter() - T0
annotation = pd.DataFrame(rows)
print(f"annotated {N:,} positions in {ANNOTATE_S:.1f} s on {N_JOBS} job(s)")
annotation.round(4)

annotated 233,339 positions in 6.4 s on -1 job(s)


,month,positions,epochs,bottom_depth_annotated,wave_height_annotated,wave_direction_annotated,seconds
0,2015-11,233339,18,1.0,1.0,1.0,6.3646


In [4]:
trips[list(THRESHOLDS)].describe().loc[["count", "mean", "std", "min", "max"]].round(2).T

,count,mean,std,min,max
bottom_depth,233339.0,44.87,162.21,2.59,4871.10
wave_height,233339.0,1.82,0.80,0.14,4.14
wave_direction,233339.0,248.15,14.25,196.77,312.95


The `*_annotated` columns above are the share of positions that received a value (1.0 when the grid brackets
all of them); a position without a value is simply never added back on that variable's account. The table
shows the range the thresholds act on.

## 4 · Compress

`compress(traj, epsilon, thresholds, circular, time_column, by, mode, filter_window, n_jobs)` takes the
points of all trips as one `GeoDataFrame` in EPSG:32630, with `by="trip_id"` compressing every trip on its own,
and returns only the retained rows with every column intact. Without `thresholds` it is plain Douglas-Peucker,
so the same call gives the baseline. Since CADP is DP *plus* added points, the DP rows are a subset of the rows
of every CADP variant, which tells us which points were kept **for context**.

Runs, each over all trips: **Douglas-Peucker**, the paper's three **single-context** variants (`depth`,
`wave_height`, `wave_direction`), and our **combined** run (all three). On the sample the combined run is also
repeated in `iterative` mode.

The vessels are compressed type by type so that each vessel type has its own timing in the summary; within one
type the trips are shared out over `N_JOBS` processes. Seconds are wall time.

In [5]:
gdf = gpd.GeoDataFrame(trips, geometry=gpd.points_from_xy(trips.x, trips.y), crs=32630)

COMMON = dict(time_column="unix_time", by="trip_id", filter_window=None, n_jobs=N_JOBS)  # data already filtered
runs = {"DP": dict(thresholds=None)}
KEY = {"DP": "dp"}
for suffix, var in VARIANTS.items():
    label = f"CADP {suffix.replace('_', ' ')}"
    runs[label] = dict(thresholds={var: THRESHOLDS[var]}, circular=[var] if var in CIRCULAR else [], mode=MODE)
    KEY[label] = f"cadp_{suffix}"
runs["CADP combined (ours)"] = dict(thresholds=THRESHOLDS, circular=CIRCULAR, mode=MODE)
KEY["CADP combined (ours)"] = "cadp_combined"
if RUN_ITERATIVE:
    runs["CADP combined iterative (ours)"] = dict(thresholds=THRESHOLDS, circular=CIRCULAR, mode="iterative")
    KEY["CADP combined iterative (ours)"] = "cadp_combined_iterative"
PAPER = [l for l in runs if l.startswith("CADP") and "(ours)" not in l]   # the paper's single-context variants

kept, rows = {k: [] for k in runs}, []
T0 = time.perf_counter()
for vtype, part in gdf.groupby("vessel_type", sort=False):
    row = {"vessel_type": vtype, "vessels": part.mmsi.nunique(), "trips": part.trip_id.nunique(),
           "points_in": len(part)}
    for label, kw in runs.items():
        t0 = time.perf_counter()
        res = compress(part, EPSILON, **COMMON, **kw)
        row[f"{KEY[label]}_points_out"] = len(res)
        row[f"{KEY[label]}_compression_ratio_pct"] = compression_ratio(len(res), len(part))
        row[f"{KEY[label]}_seconds"] = time.perf_counter() - t0
        kept[label].append(res.index.to_numpy())
    rows.append(row)
COMPRESS_S = time.perf_counter() - T0
kept = {k: np.sort(np.concatenate(v)) for k, v in kept.items()}   # the index is trips' row number
del gdf
print(f"compressed {N:,} points in {COMPRESS_S:.1f} s on {N_JOBS} job(s)")
for label in runs:
    print(f"  {label:<32} keeps {len(kept[label]):>10,}   R = {compression_ratio(len(kept[label]), N):6.2f} %")

compressed 233,339 points in 3.3 s on -1 job(s)
  DP                               keeps      1,646   R =  99.29 %
  CADP depth                       keeps      3,578   R =  98.47 %
  CADP wave height                 keeps     46,054   R =  80.26 %
  CADP wave direction              keeps     66,212   R =  71.62 %
  CADP combined (ours)             keeps     78,605   R =  66.31 %
  CADP combined iterative (ours)   keeps      3,195   R =  98.63 %


## 5 · Save

The compressed rows are rows of the clean data: no point is moved (unlike 2stage-CPLS, CADP has no stay
collapsing), so every column keeps its original value. Written under `results/04_cadp_compression/<DATA>/`:

* `compressed_ais_depth.parquet`, `compressed_ais_wave_height.parquet`, `compressed_ais_wave_direction.parquet`:
  the paper's three variants. Each has the same columns as notebook 03's output (`mmsi`, `trip_id`,
  `unix_time`, `x`, `y`, `lon`, `lat`, `speed_over_ground`, `course_over_ground`, `vessel_type`,
  `navigational_status`, `true_heading`, `rate_of_turn`, `ship_type_code`; `trip_id` is the same as in notebook 03's
  `trips_ais.parquet`), plus the three annotated context columns and `kept_by_context`, True on the points CADP added
  to what Douglas-Peucker keeps;
* `compressed_ais_combined.parquet`: our addition, all three contexts at once, same columns;
* `summary.csv`: points in and out, compression ratio and seconds, per vessel type and in total, for
  Douglas-Peucker and every CADP run;
* `context_fidelity.csv`: the check of section 7.

In [6]:
masks = {}
for label in runs:
    m = np.zeros(N, bool); m[kept[label]] = True
    masks[label] = m
is_dp = masks["DP"]

FILES = {f"CADP {suffix.replace('_', ' ')}": f"compressed_ais_{suffix}.parquet" for suffix in VARIANTS}
FILES["CADP combined (ours)"] = "compressed_ais_combined.parquet"
for label, name in FILES.items():
    out = trips.iloc[kept[label]].copy()
    out["kept_by_context"] = ~is_dp[kept[label]]
    out.to_parquet(OUT / name, index=False)
    del out

summary = pd.DataFrame(rows).sort_values("points_in", ascending=False)
total = {"vessel_type": "ALL", "vessels": trips.mmsi.nunique(), "trips": N_TRIPS, "points_in": N}
for label in runs:
    total[f"{KEY[label]}_points_out"] = len(kept[label])
    total[f"{KEY[label]}_compression_ratio_pct"] = compression_ratio(len(kept[label]), N)
    total[f"{KEY[label]}_seconds"] = summary[f"{KEY[label]}_seconds"].sum()
summary = pd.concat([summary, pd.DataFrame([total])], ignore_index=True)
summary["annotate_seconds"] = ANNOTATE_S      # the same for every row: one pass over all positions
summary["split_trips_seconds"] = SPLIT_S
summary["n_jobs"] = N_JOBS
summary.to_csv(OUT / "summary.csv", index=False)
for f in sorted(OUT.iterdir()):
    print(f"{f.name:40s} {f.stat().st_size / 1e6:8.2f} MB")

compressed_ais_combined.parquet              5.29 MB
compressed_ais_depth.parquet                 0.29 MB
compressed_ais_wave_direction.parquet        4.48 MB
compressed_ais_wave_height.parquet           3.09 MB
context_fidelity.csv                         0.00 MB
summary.csv                                  0.00 MB
trips_ais.parquet                            7.76 MB


## 6 · Results

**Compression ratio** R is the percentage of points removed (Eq. 9 of the paper):
`R = (1 - points out / points in) * 100`. First overall and by vessel type, for plain Douglas-Peucker, the paper's
three CADP variants and our combined run, with the ratio of **2stage-CPLS** from notebook 03 next to them when that
notebook has been run for this dataset. The methods do different things: DP only drops points that lie near the
line, CADP additionally keeps every point where *one* context departs from its interpolation, the combined run
every point where *any* of the three does, and 2stage-CPLS keeps points for topology and stays; so the ratios say
what each criterion costs, not which one is better. For orientation, the paper reports R = 70-73 % for CADP with
wave direction, 76.78 % for DP and 72.08 % for 2stage-CPLS on its own data; this data and period differ, so the
numbers here are not a reproduction of those.

In [7]:
rcols = [f"{KEY[l]}_compression_ratio_pct" for l in runs if "iterative" not in l]
show = summary[["vessel_type", "vessels", "trips", "points_in"] + rcols].copy()
show.columns = ["vessel_type", "vessels", "trips", "points_in"] + [l + " R%" for l in runs if "iterative" not in l]
nb03_ok = False
if (NB03 / "summary.csv").is_file():
    _s3 = pd.read_csv(NB03 / "summary.csv")
    nb03_ok = int(_s3.loc[_s3.vessel_type == "ALL", "points_in"].iloc[0]) == N   # same positions in only
if nb03_ok:
    s3 = pd.read_csv(NB03 / "summary.csv")[["vessel_type", "points_in", "compression_ratio_pct"]]
    s3 = s3.rename(columns={"points_in": "nb03_points_in", "compression_ratio_pct": "2stage-CPLS R%"})
    show = show.merge(s3, on="vessel_type", how="left")
    print("2stage-CPLS ratios read from", (NB03 / "summary.csv").relative_to(ROOT).as_posix(),
          "- same input positions:", bool((show.nb03_points_in == show.points_in).all()))
    show = show.drop(columns="nb03_points_in")
else:
    print("notebook 03 has not been run on these positions: no 2stage-CPLS column")
rcol_names = [c for c in show.columns if c.endswith("R%")]
print(f"overall, {N:,} positions in:")
for label in runs:
    n = len(kept[label])
    print(f"  {label:<32} {n:>10,} points   R = {compression_ratio(n, N):6.2f} %   "
          f"added for context: {n - len(kept['DP']):>9,}")
show.round({c: 2 for c in rcol_names})

2stage-CPLS ratios read from results/03_two_stage_cpls_compression/sample/summary.csv - same input positions: True
overall, 233,339 positions in:
  DP                                    1,646 points   R =  99.29 %   added for context:         0
  CADP depth                            3,578 points   R =  98.47 %   added for context:     1,932
  CADP wave height                     46,054 points   R =  80.26 %   added for context:    44,408
  CADP wave direction                  66,212 points   R =  71.62 %   added for context:    64,566
  CADP combined (ours)                 78,605 points   R =  66.31 %   added for context:    76,959
  CADP combined iterative (ours)        3,195 points   R =  98.63 %   added for context:     1,549


,vessel_type,vessels,trips,points_in,DP R%,CADP depth R%,CADP wave height R%,CADP wave direction R%,CADP combined (ours) R%,2stage-CPLS R%
0,Passenger,14,36,54801,99.63,99.63,77.13,71.80,63.26,88.07
1,Cargo,76,173,53059,99.25,96.76,81.83,76.96,69.55,82.04
2,Fishing,69,128,37022,99.06,99.06,66.61,55.08,49.22,83.36
3,SAR,4,4,19572,99.94,99.94,80.24,68.70,68.47,98.91
4,Tug,2,4,17061,99.91,99.91,99.91,93.41,93.41,98.87
5,Unknown,25,44,16733,99.00,99.00,85.89,71.16,69.44,71.69
6,Tanker,25,81,16117,98.78,96.41,92.15,74.62,72.02,81.26
7,Sailing Vessel,13,30,6147,98.19,94.44,80.92,67.81,59.05,87.02
8,Other,14,30,4741,98.16,98.16,69.06,57.27,55.68,84.08
9,Pilot Vessel,1,7,4601,99.04,99.04,82.79,81.13,75.79,82.61


In [8]:
types = show[show.vessel_type != "ALL"].sort_values("points_in", ascending=False).head(12)
alls = show[show.vessel_type == "ALL"]
chart = pd.concat([alls, types]).rename(columns={"vessel_type": "group"})
chart["group"] = chart.group.replace({"ALL": "ALL (every type)"})
long = chart.melt(id_vars="group", value_vars=rcol_names, var_name="method", value_name="R (%)")
long["method"] = long.method.str.replace(" R%", "", regex=False)
long["group"] = pd.Categorical(long.group, categories=chart.group.tolist(), ordered=True)
long.hvplot.bar(x="group", y="R (%)", by="method", rot=35, width=1000, height=420, ylim=(0, 100),
                legend="bottom", xlabel="", title="compression ratio: overall and for the 12 vessel types with most positions")

:Bars   [group,method]   (R (%))

**What to look at.** Every CADP variant's ratio is at most Douglas-Peucker's: each point it adds is a point DP
dropped. The three single-context variants differ in how often their variable drifts past its threshold along a
DP segment; the combined run keeps at least the union of the three, so it is always the lowest of the CADP bars.
Wave height at 0.15 m and direction at 1.5 degrees are tight compared with the variation of a 3-hourly field along a
voyage, so wherever the sea state changes along a leg many points are added back (section 7 shows this on single
vessels); bottom depth at 80 m changes only over a steep shelf and adds few points. A vessel that hardly moves meets
hardly any change of context and gets few added points.

In [9]:
if RUN_ITERATIVE:
    order = ["DP", "CADP combined (ours)", "CADP combined iterative (ours)"]
    allrow = summary[summary.vessel_type == "ALL"].iloc[0]
    cmp = pd.DataFrame({"method": order,
                        "points_out": [len(kept[k]) for k in order],
                        "R (%)": [compression_ratio(len(kept[k]), N) for k in order],
                        "seconds": [allrow[f"{KEY[k]}_seconds"] for k in order]})
    print("the same trips and the three thresholds together, the two modes of CADP:")
    display(cmp.round(2))
else:
    print("CADP combined iterative not run (RUN_ITERATIVE = False)")

the same trips and the three thresholds together, the two modes of CADP:


,method,points_out,R (%),seconds
0,DP,1646,99.29,0.48
1,CADP combined (ours),78605,66.31,0.54
2,CADP combined iterative (ours),3195,98.63,0.57


In `iterative` mode only each segment's worst point is added per round and the errors are measured again, so
it keeps the apex of a bow in the context curve and usually finds the other points on the bow now within
tolerance: far fewer points, and a bound on the final error (next section), at the cost of several rounds.
`post` is the published method and the one used for every saved file.

## 7 · Vessels before and after, and the fidelity of the context

**A few vessels.** For the four vessel types with most positions, the trip with the most positions among those
that span more than 20 km: the original track in grey, the points Douglas-Peucker keeps in orange, and the points
each CADP variant adds in its own colour (click a legend entry to mute a layer). The lower panels show the wave height
met along each trip with the DP points and the points added by the wave-height variant. Zoom, pan and hover to read a
point's time and wave height.

In [10]:
coast = load_coastline(PRE).to_crs(4326)
MOVING_KM = 20.0
MAX_PTS = 2_500                      # at most this many points per vessel and layer in a figure
ADDED = {"CADP depth": "#1a7f37", "CADP wave height": "#0969da", "CADP wave direction": "#8250df"}

top_types = summary[summary.vessel_type != "ALL"].vessel_type.head(4).tolist()
chosen = []
for vt in top_types:
    sizes_vt = trips[trips.vessel_type == vt].groupby("trip_id").size().sort_values(ascending=False)
    for tid in sizes_vt.index[:200]:
        g = trips[trips.trip_id == tid]
        if np.hypot(np.ptp(g.x), np.ptp(g.y)) / 1000 > MOVING_KM:
            chosen.append((vt, tid))
            break

def thin(df, n=MAX_PTS):
    return df.iloc[::max(1, len(df) // n)]

maps, series = [], []
for vt, tid in chosen:
    g = trips[trips.trip_id == tid].copy()
    g["time"] = pd.to_datetime(g.unix_time, unit="s")
    dp = g[is_dp[g.index]]
    n_each = {l: int((masks[l][g.index] & ~is_dp[g.index]).sum()) for l in ADDED}
    title = f"{vt}, vessel {g.mmsi.iloc[0]}: {len(g):,} points, DP keeps {len(dp):,}"
    pad = 0.08
    box = coast.cx[g.lon.min() - pad:g.lon.max() + pad, g.lat.min() - pad:g.lat.max() + pad]
    m = thin(g).hvplot.line(x="lon", y="lat", geo=True, tiles="CartoLight", color="#8c959f", line_width=2,
                            title=title, width=430, height=380)
    if len(box):
        m = m * box.hvplot.polygons(geo=True, color="#ece8dc", line_color="#a8a39a", alpha=0.6)
    for label, color in ADDED.items():
        add = g[masks[label][g.index] & ~is_dp[g.index]]
        if len(add):
            m = m * thin(add).hvplot.points(x="lon", y="lat", geo=True, color=color, size=12,
                                            hover_cols=["time", "wave_height"],
                                            label=f"{label.replace('CADP ', '')} (+{n_each[label]:,})")
    m = m * dp.hvplot.points(x="lon", y="lat", geo=True, color="#e16f24", size=30,
                             hover_cols=["time", "wave_height"], label="Douglas-Peucker")
    maps.append(m.opts(legend_position="bottom_left", active_tools=["wheel_zoom"]))
    add_wh = g[masks["CADP wave height"][g.index] & ~is_dp[g.index]]
    s_ = thin(g).hvplot.line(x="time", y="wave_height", color="#8c959f", width=430, height=220, ylabel="wave height (m)")
    if len(add_wh):
        s_ = s_ * thin(add_wh).hvplot.scatter(x="time", y="wave_height", color="#0969da", size=18)
    series.append(s_ * dp.hvplot.scatter(x="time", y="wave_height", color="#e16f24", size=30))
print(f"{len(chosen)} vessels shown; at most {MAX_PTS:,} points per layer are drawn")
hv.Layout(maps + series).cols(len(maps)).opts(shared_axes=False)

3 vessels shown; at most 2,500 points per layer are drawn


:Layout
   .Overlay.I   :Overlay
      .WMTS.I                                                                    :WMTS   [Longitude,Latitude]
      .Curve.I                                                                   :Curve   [lon]   (lat)
      .Polygons.I                                                                :Polygons   [Longitude,Latitude]
      .Points.Wave_height_left_parenthesis_plus_1_comma_159_right_parenthesis    :Points   [lon,lat]   (time,wave_height)
      .Points.Wave_direction_left_parenthesis_plus_1_comma_800_right_parenthesis :Points   [lon,lat]   (time,wave_height)
      .Points.Douglas_hyphen_minus_Peucker                                       :Points   [lon,lat]   (time,wave_height)
   .Overlay.II  :Overlay
      .WMTS.I                                                            :WMTS   [Longitude,Latitude]
      .Curve.I                                                           :Curve   [lon]   (lat)
      .Polygons.I                                                        :Polygons   [Longitude,Latitude]
      .Points.Wave_height_left_parenthesis_plus_565_right_parenthesis    :Points   [lon,lat]   (time,wave_height)
      .Points.Wave_direction_left_parenthesis_plus_907_right_parenthesis :Points   [lon,lat]   (time,wave_height)
      .Points.Douglas_hyphen_minus_Peucker                               :Points   [lon,lat]   (time,wave_height)
   .Overlay.III :Overlay
      .WMTS.I                                                                    :WMTS   [Longitude,Latitude]
      .Curve.I                                                                   :Curve   [lon]   (lat)
      .Polygons.I                                                                :Polygons   [Longitude,Latitude]
      .Points.Wave_height_left_parenthesis_plus_2_comma_306_right_parenthesis    :Points   [lon,lat]   (time,wave_height)
      .Points.Wave_direction_left_parenthesis_plus_3_comma_109_right_parenthesis :Points   [lon,lat]   (time,wave_height)
      .Points.Douglas_hyphen_minus_Peucker                                       :Points   [lon,lat]   (time,wave_height)
   .Overlay.IV  :Overlay
      .Curve.I    :Curve   [time]   (wave_height)
      .Scatter.I  :Scatter   [time]   (wave_height)
      .Scatter.II :Scatter   [time]   (wave_height)
   .Overlay.V   :Overlay
      .Curve.I    :Curve   [time]   (wave_height)
      .Scatter.I  :Scatter   [time]   (wave_height)
      .Scatter.II :Scatter   [time]   (wave_height)
   .Overlay.VI  :Overlay
      .Curve.I    :Curve   [time]   (wave_height)
      .Scatter.I  :Scatter   [time]   (wave_height)
      .Scatter.II :Scatter   [time]   (wave_height)

**Fidelity.** CADP's promise is about the dropped points: how far is their context from the interpolation
between their *final* neighbours, the quantity the thresholds are about? `catraj.compression.context_errors`
measures it for every dropped point, for all three variables and every run. In `post` mode the errors are measured
against the **DP** neighbours, and once points are added those are no longer the neighbours, so the bound holds on the
DP segments, not on the final result. The table gives the RMSE, the largest error and the share of dropped points over
the threshold. A single-context variant is expected to be good on its own variable and no better than DP on the others.
Trips are checked in parallel, one task each (on the full archive a random 3,000 trips, the same for every run, so
the check stays small).

In [11]:
rng = np.random.default_rng(0)
ids = trips.trip_id.unique()
if len(ids) > 3_000:
    ids = np.sort(rng.choice(ids, 3_000, replace=False))
sub = trips[trips.trip_id.isin(ids)]
keep_sets = masks
groups = [g for _, g in sub.groupby("trip_id", sort=False)]
tasks = [(g.unix_time.to_numpy(float), {v: g[v].to_numpy(float) for v in THRESHOLDS},
          {k: m[g.index] for k, m in keep_sets.items()}) for g in groups]
circular_vars = set(CIRCULAR)

def fidelity(task):
    t, ctx, ms = task
    out = {}
    for label, m in ms.items():
        for v, c in ctx.items():
            e = context_errors(t, c, m, circular=v in circular_vars)[~m]
            out[(label, v)] = e[np.isfinite(e)]
    return out

t0 = time.perf_counter()
per_trip = parallel_map(fidelity, tasks, N_JOBS, [len(t[0]) for t in tasks])
rows_f = []
for label in keep_sets:
    for v, thr in THRESHOLDS.items():
        e = np.concatenate([p[(label, v)] for p in per_trip])
        rows_f.append({"method": label, "variable": v, "threshold": thr,
                       "rmse": float(np.sqrt(np.mean(e ** 2))) if e.size else 0.0,
                       "max": float(e.max()) if e.size else 0.0,
                       "dropped_over_threshold_pct": 100 * float((e > thr).mean()) if e.size else 0.0})
fid = pd.DataFrame(rows_f)
fid.to_csv(OUT / "context_fidelity.csv", index=False)
print(f"{len(groups):,} trips, {len(sub):,} positions, checked in {time.perf_counter() - t0:.1f} s on {N_JOBS} job(s)")
fid.round(3)

563 trips, 233,339 positions, checked in 0.5 s on -1 job(s)


,method,variable,threshold,rmse,max,dropped_over_threshold_pct
0,DP,bottom_depth,80.00,58.123,1472.726,0.834
1,DP,wave_height,0.15,0.160,1.645,19.167
2,DP,wave_direction,1.50,3.325,43.590,27.867
3,CADP depth,bottom_depth,80.00,4.808,148.534,0.019
4,CADP depth,wave_height,0.15,0.160,1.645,19.328
5,CADP depth,wave_direction,1.50,3.339,43.590,28.100
6,CADP wave height,bottom_depth,80.00,64.546,1472.726,1.032
7,CADP wave height,wave_height,0.15,0.058,0.467,2.261
8,CADP wave height,wave_direction,1.50,1.315,23.400,12.715
9,CADP wave direction,bottom_depth,80.00,68.098,1472.726,1.149


**Reading the table.** Douglas-Peucker ignores the context, so a share of the points it drops are over the
thresholds, by far more than the thresholds. Each single-context variant puts back the points over *its* threshold
against the DP neighbours, which cuts the error of its own variable; what remains is caused by the added points
changing the neighbours. The combined run does so for all three; `iterative` mode, when run, has a maximum within each
threshold by construction.

## 8 · Summary

The cell below prints the headline numbers, which are also in `summary.csv`.

In [12]:
tot = summary[summary.vessel_type == "ALL"].iloc[0]
print(f"dataset: {DATA}   trips: {N_TRIPS:,}   positions in: {N:,}   vessels: {trips.mmsi.nunique():,}")
print(f"epsilon {EPSILON:g} m; thresholds " + ", ".join(f"{k} {v:g}" for k, v in THRESHOLDS.items()) + f"; mode {MODE}")
for label in runs:
    print(f"{label:<32}: {int(tot[KEY[label] + '_points_out']):>10,} points   R = {tot[KEY[label] + '_compression_ratio_pct']:.2f} %")
if nb03_ok:
    s3 = pd.read_csv(NB03 / "summary.csv"); s3 = s3[s3.vessel_type == "ALL"].iloc[0]
    print(f"{'2stage-CPLS (notebook 03)':<32}: {int(s3.points_out):>10,} points   R = {s3.compression_ratio_pct:.2f} %")
print(f"seconds: annotate {ANNOTATE_S:.1f}, compress ({len(runs)} runs) {COMPRESS_S:.1f}, on {N_JOBS} job(s)")

dataset: sample   trips: 563   positions in: 233,339   vessels: 253
epsilon 1000 m; thresholds bottom_depth 80, wave_height 0.15, wave_direction 1.5; mode post
DP                              :      1,646 points   R = 99.29 %
CADP depth                      :      3,578 points   R = 98.47 %
CADP wave height                :     46,054 points   R = 80.26 %
CADP wave direction             :     66,212 points   R = 71.62 %
CADP combined (ours)            :     78,605 points   R = 66.31 %
CADP combined iterative (ours)  :      3,195 points   R = 98.63 %
2stage-CPLS (notebook 03)       :     32,950 points   R = 85.88 %
seconds: annotate 6.4, compress (6 runs) 3.3, on -1 job(s)
